# Model Evaluation
Evaluate the saved model on a held-out dataset using metrics appropriate to your target type. Keep evaluation data separate from training data.

In [ ]:
from pathlib import Path
import joblib
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
MODEL_PATH = PROJECT_ROOT / 'models' / 'irrigation_model.joblib'
PROCESSED_DATA_DIR = PROJECT_ROOT / 'data' / 'processed'

In [ ]:
TARGET_COLUMN = 'replace_with_target'
processed_files = sorted(PROCESSED_DATA_DIR.glob('*.csv'))
if not MODEL_PATH.is_file() or not processed_files:
    raise FileNotFoundError('Train a model and create processed data before evaluation.')
data = pd.read_csv(processed_files[0])
if TARGET_COLUMN not in data.columns:
    raise ValueError(f'Set TARGET_COLUMN to a column in the dataset: {list(data.columns)}')
X = data.drop(columns=[TARGET_COLUMN]).select_dtypes(include='number')
y = data[TARGET_COLUMN]
_, X_test, _, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = joblib.load(MODEL_PATH)
predictions = model.predict(X_test)
if y.dtype.kind in 'if':
    print({'mae': mean_absolute_error(y_test, predictions), 'rmse': mean_squared_error(y_test, predictions) ** 0.5, 'r2': r2_score(y_test, predictions)})
else:
    print(f'accuracy: {accuracy_score(y_test, predictions):.3f}')
    print(classification_report(y_test, predictions))